In [1]:
import time
start_time = time.time()
%matplotlib qt
mpl_time = time.time()
import pyoptflight as pof
pof_main_time = time.time()
from pyoptflight import initialize as optinit
pof_init_time = time.time()
from pyoptflight import plotting as optplot
pof_plotting_time = time.time()
import numpy as np
np_time = time.time()

print(f'mpl: {mpl_time - start_time}')
print(f'pof main: {pof_main_time - mpl_time}')
print(f'pof init: {pof_init_time - pof_main_time}')
print(f'pof plotting: {pof_plotting_time - pof_init_time}')
print(f'np: {np_time - pof_plotting_time}')

mpl: 0.4760470390319824
pof main: 1.0689375400543213
pof init: 4.982948303222656e-05
pof plotting: 3.6716461181640625e-05
np: 3.337860107421875e-05


In [2]:
kerbin = pof.Body.load("Kerbin")
# kerbin.omega_0 = 0
# vehicle = pof.Vehicle.load('MintOCMulti')
# vehicle = pof.Vehicle.load('MintOCSingle')
vehicle = pof.Vehicle.load("TestVehicle")

# vehicle[0].m_f = 8.4
# vehicle[1].m_0 = 8.3

config = pof.SolverConfig(landing=False, 
                          T_min      = 1,
                          T_max      = 1000,
                          max_iter   = 1500, 
                          solver_tol = 1e-3, 
                          N          = 50, 
                          T_init     = 300,
                          integration_method = 'RK4',
                          aero_model = 'axial_normal')

x0 = pof.LatLngBound(lat=0, lng=0, alt=0, vel = 1e-6, ERA0=0)

xf = pof.KeplerianBound(i    = np.deg2rad(0),
                        Ω    = np.deg2rad(0),
                        ω    = 0,
                        ha   = 80,
                        hp   = 80,
                        body = kerbin)

msolver = pof.Solver(kerbin, vehicle, config, x0, xf)

In [3]:
msolver.create_nlp()

In [ ]:
msolver.update_constraints(constraint_names='max_tau', new_values=0.20, new_enables=True)
# msolver.update_constraints(constraint_names='max_body_rate_y', new_values=np.deg2rad(5), new_enables=True)
# msolver.update_constraints(constraint_names='max_body_rate_z', new_values=np.deg2rad(5), new_enables=True)
# msolver.update_constraints(constraint_names='f_min', new_values=0.15, new_enables=True)
msolver.update_constraints(constraint_names='max_alpha', new_values=np.deg2rad(90), new_enables=True)
# msolver.update_constraints(constraint_names='max_q', new_values=0.2, new_enables=True)
msolver.context.constraints

In [ ]:
msolver.context.config.aero_model = 'lift_drag'

In [ ]:
import casadi as ca
import numpy as np

# Setup (same as above)
xgrid = np.linspace(-5,5,11)
ygrid = np.linspace(-4,4,9)
X,Y = np.meshgrid(xgrid,ygrid,indexing='ij')
R_val = np.sqrt(5*X**2 + Y**2)+ 1
data_val = np.sin(R_val)/R_val
data_flat_val = data_val.ravel(order='F')
lut_bspline = ca.interpolant('my_bspline_lut','bspline',[xgrid,ygrid],data_flat_val)

# Define MX symbols for the Function's internal representation
mx_arg1 = ca.MX.sym('mx_arg1')
mx_arg2 = ca.MX.sym('mx_arg2')
mx_point_for_lut = ca.vertcat(mx_arg1, mx_arg2)

# Create a CasADi Function that wraps the MX-based interpolant
# This function takes MX inputs and produces an MX output internally
bspline_mx_function = ca.Function('bspline_mx_function',
                                  [mx_arg1, mx_arg2],  # Inputs to this wrapper function
                                  [lut_bspline(mx_point_for_lut)]) # Expression using the interpolant

# Now, define your SX symbols for the external call
sx_var1 = ca.SX.sym('sx_var1')
sx_var2 = ca.SX.sym('sx_var2')

# Call the MX-based function with SX inputs
# CasADi handles SX -> MX -> SX conversion
print("\nCalling MX-based B-spline interpolant Function with SX inputs:")
sx_output = bspline_mx_function(sx_var1, sx_var2)
print(sx_output)

# sx_output is now an SX expression that can be used in other SX calculations
test_sx_func = ca.Function('test_sx_func', [sx_var1, sx_var2], [sx_output])
print("Evaluation of the SX-wrapped result at (0.5, 1.0):")
print(test_sx_func(0.5, 1.0)) # Should match the MX result

In [ ]:
msolver.initialize_from_func(pof.gravity_turn, opts={'skew':True, 'integration_method': 'RK4'})

In [4]:
msolver.initialize_from_func(pof.cubic_bezier_spline, opts={'spacing': 'dV'})

The minimum cost is 0.006308418983111963 rad


In [5]:
msolver.solve_nlp()


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:     8327
Number of nonzeros in inequality constraint Jacobian.:      317
Number of nonzeros in Lagrangian Hessian.............:     6438

Option nlp_scaling_method selected as user-scaling, but no user-scaling available, or it cannot be computed.
Exception of type: OPTION_INVALID in file "Interfaces/IpTNLPAdapter.cpp" at line 2022:
 Exception message: User scaling chosen, but get_scaling_parameters returned false.

EXIT: Invalid option encountered.
   nlpsolver  :   t_proc      (av

In [ ]:
fig = pof.plot_flight_data(msolver, 
                           plot_on_nodes=True, 
                           use_radians=False, 
                        #    include=['f'],
                           )

In [ ]:
pof.plot_solutions(msolver, colorscale=None, markers='ctrl', show_actual_orbit=True, show_target_orbit=True, indices=[-1])

In [ ]:
N = [50, 50]
cum_N = np.concatenate(([0], np.cumsum(N)))
i=1
k = np.searchsorted(cum_N[:-1], i, side='right') -1
k

In [ ]:
context = msolver.context
fixed_states = pof.initialize._fix_states(context, x0, xf)
kep_state = pof.functions.state_to_kep(np.concatenate([fixed_states['xf']['pos'], fixed_states['xf']['vel']]), kerbin.mu)[0:5]
fig = pof.plot_celestial(kerbin)
pof.plot_orbit(fig, *kep_state, kerbin.mu)
# fig.show()
print(fixed_states['xf'])

In [ ]:
msolver.stage_sols[-1][-1].X[-1]

In [ ]:
kerbin.atm.T